# Comprobaciones de la revisión final (D59)
**Proyecto:** Predicción de matrícula y continuidad estudiantil · Unidad Educativa LEMAS
**Integrantes:** Guillermo Granizo y José Ulloa · Maestría en Inteligencia Artificial (UEES)

Una revisión independiente de la entrega planteó cuatro dudas sobre la evaluación. Este cuaderno las responde con datos, sin cambiar el sistema elegido:

| Duda | Qué se hace | Cohortes |
|---|---|---|
| El análisis exploratorio (cuaderno 01) usó todas las cohortes, así que D2 se diseñó viendo C4 y C5 | Se repiten las tasas por señal **solo con C1–C3** y se comprueba si el orden de las señales, que es lo único que define a D2, sale igual | C1–C3 |
| El Lift@k de D2 depende del sorteo de empates | Distribución del Lift@k con 500 sorteos | C4 |
| Los IC de D2 y de D se solapan: ¿D2 es mejor que D? | IC 95 % de la diferencia D2 − D, pareada, por bootstrap de representantes y con sorteo nuevo en cada remuestra | C4 |
| El azar da un Lift@k de 1,08 y no de 1,00 | Azar por estudiante (la referencia publicada) frente a azar uniforme por familia | C4 |

Además se calculan los **estadísticos descriptivos** de las variables numéricas por cohorte (sin la etiqueta).

**Reglas:** C5 no interviene, salvo que se marque `INCLUIR_C5` (ver la sección 5). Solo se guardan agregados; las celdas con menos de 5 estudiantes o eventos se suprimen.

In [ ]:
# @title 0 · Preparar el entorno (ejecutar primero)
# Si el repositorio ya está en GitHub, pegue su URL; si no, se pedirá subir el .zip.
REPO_URL = "https://github.com/ggranizo2507/prediccion-matricula-lemas.git"  # @param {type:"string"}

import importlib.util, os, subprocess, sys, zipfile
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    RAIZ = Path("/content/prediccion-matricula-lemas")
    if not RAIZ.exists():
        if REPO_URL:
            subprocess.run(["git", "clone", "-q", REPO_URL, str(RAIZ)], check=True)
        else:
            from google.colab import files
            print("Suba el archivo prediccion-matricula-lemas.zip")
            nombre = next(iter(files.upload()))
            zipfile.ZipFile(nombre).extractall("/content")
else:
    RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
REQUERIDOS = {'pandas': 'pandas', 'sklearn': 'scikit-learn', 'yaml': 'PyYAML',
              'optuna': 'optuna', 'matplotlib': 'matplotlib'}  # módulo -> paquete pip
faltan = [pip for modulo, pip in REQUERIDOS.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=True)
print("Entorno:", "Google Colab" if EN_COLAB else "local")

In [ ]:
# @title Cargar la base
FUENTE = "sintetica"  # @param ["sintetica", "real"]
SUBIR_DE_NUEVO = False  # @param {type:"boolean"}
INCLUIR_C5 = False  # @param {type:"boolean"}

import json, logging, warnings
warnings.filterwarnings("ignore", message=".*IProgress.*")
import pandas as pd
from IPython.display import display
from src.data_processing import construir_dataset, limpiar_base
from src.utils import cargar_config, configurar_logging, obtener_base_seud

configurar_logging(logging.WARNING)
config = cargar_config()
SEMILLA = config["proyecto"]["semilla"]
MIN = config["privacidad"]["min_celda"]
SORTEOS = config["revision"]["sorteos"]
REPETICIONES = config["revision"]["repeticiones_bootstrap"]
ruta = RAIZ / config["rutas"]["base_real" if FUENTE == "real" else "base_sintetica"]
crudo = (obtener_base_seud(ruta, EN_COLAB, SUBIR_DE_NUEVO) if FUENTE == "real"
         else pd.read_csv(ruta, dtype=str))
dataset, _ = construir_dataset(limpiar_base(crudo, config), config)
MET = RAIZ / config["rutas"]["metricas"]; MET.mkdir(parents=True, exist_ok=True)
decision = json.loads((MET / f"decision_c1_{FUENTE}.json").read_text(encoding="utf-8"))
K = decision["k_por_sede"]
C4 = dataset[dataset["rol"] == "seleccion"]
print(f"Fuente: {FUENTE} | estudiantes modelables: {len(dataset)} | k por sede: {K}")
print("C5:", "se usa en la sección 5 (solo medición, sin elegir nada)" if INCLUIR_C5 else "no se usa")
if FUENTE == "sintetica":
    print("⚠️ Datos SINTÉTICOS: sirven para probar el código; sus conclusiones no describen a LEMAS.")

## 1. ¿D2 habría salido igual sin mirar C4 ni C5?
D2 ordena primero por pago tardío de la matrícula anterior, luego por reserva extraordinaria y, dentro de cada grupo, por atrasos. Ese orden es su única "decisión". Aquí se calcula, con las cohortes anteriores a la selección, la tasa de no matrícula con y sin cada señal y su **riesgo relativo** (tasa con la señal ÷ tasa sin ella). Para comparar señales binarias, los atrasos se resumen como "1 o más". Si el orden por riesgo relativo es el mismo, D2 se habría construido igual sin ver C4 ni C5.

In [ ]:
# @title Tasas por señal con C1–C3
from src import revision as rv

PERIODOS = {"C1–C3": ["C1", "C2", "C3"], "C2–C3 (entrenamiento)": ["C2", "C3"],
            "C1": ["C1"], "C2": ["C2"], "C3": ["C3"]}
SENALES = rv.senales_por_periodo(dataset, PERIODOS, MIN)
ORDEN = rv.orden_senales(SENALES)
display(SENALES[SENALES["periodo"] == "C1–C3"])
display(ORDEN)

## 2 y 3. Sorteo de desempate y diferencia D2 − D en C4
Muchas familias empatan en el puntaje de D2 (por ejemplo, todas las que pagaron en plazo, con reserva ordinaria y sin atrasos). Los empates se resuelven al azar con una semilla fija, la misma que usa la aplicación. Aquí se repite el cálculo con 500 semillas. Después se estima el IC 95 % de la diferencia de Lift@k entre D2 y D sobre las **mismas** remuestras de representantes, con un sorteo nuevo en cada una.

In [ ]:
# @title Sorteos y diferencia pareada en C4
from src.reglas import puntaje_d, puntaje_d2

SORTEOS_C4 = rv.sorteos_desempate(C4, puntaje_d2(C4), K, range(SORTEOS))
RESUMEN_C4 = rv.resumen_sorteos(SORTEOS_C4, SEMILLA)
PAREADA_C4 = rv.diferencia_pareada(C4, puntaje_d2(C4), puntaje_d(C4), K, SEMILLA, REPETICIONES)
display(pd.Series(RESUMEN_C4, name="D2 en C4"))
display(pd.Series(PAREADA_C4, name="D2 − D en C4"))

## 4. ¿Por qué el azar no da 1,00?
La referencia publicada sortea un número por **estudiante** y la familia toma el máximo, igual que con cualquier puntaje. Así, una familia con dos hijos tiene dos números en el sorteo y más probabilidad de entrar en la lista. Si esas familias tienen más casos de no matrícula, el Lift@k del azar queda por encima de 1. Con un sorteo uniforme **por familia**, el Lift@k esperado es 1.

In [ ]:
# @title Azar por estudiante frente a azar por familia
AZAR_C4 = rv.azar_dos_versiones(C4, K, SEMILLA, 500)
display(pd.Series(AZAR_C4, name="azar en C4"))

## 5. (Opcional) Las mismas mediciones en C5
C5 se usó una sola vez, el 01-oct-2026, con el sistema congelado. Estas mediciones **no eligen ni cambian nada**: D y D2 son reglas fijas, sin parámetros que ajustar. Solo describen la incertidumbre del resultado ya publicado (el sorteo de empates y la diferencia D2 − D). Se ejecutan únicamente si el equipo marca `INCLUIR_C5` y lo registra en la planificación.

In [ ]:
# @title Sorteos y diferencia pareada en C5 (solo si INCLUIR_C5)
RESUMEN_C5, PAREADA_C5 = None, None
if INCLUIR_C5:
    C5 = dataset[dataset["rol"] == "prueba"]
    RESUMEN_C5 = rv.resumen_sorteos(rv.sorteos_desempate(C5, puntaje_d2(C5), K, range(SORTEOS)), SEMILLA)
    PAREADA_C5 = rv.diferencia_pareada(C5, puntaje_d2(C5), puntaje_d(C5), K, SEMILLA, REPETICIONES)
    display(pd.Series(RESUMEN_C5, name="D2 en C5"))
    display(pd.Series(PAREADA_C5, name="D2 − D en C5"))
else:
    print("C5 no se usa. Para medir la incertidumbre del resultado final, marque INCLUIR_C5.")

## 6. Estadísticos descriptivos
Media, desviación estándar y percentiles 5, 25, 50, 75 y 95 de las variables numéricas, por cohorte y en total. Se informan los percentiles 5 y 95 en lugar del mínimo y el máximo, que serían el valor de un solo estudiante.

In [ ]:
# @title Estadísticos descriptivos por cohorte
DESCRIPTIVOS = rv.descriptivos(dataset, min_celda=MIN)
display(DESCRIPTIVOS[DESCRIPTIVOS["cohorte"] == "Total"])

## Conclusiones y archivos

In [ ]:
# @title Conclusiones y guardado
CONCLUSIONES = rv.conclusiones(ORDEN, RESUMEN_C4, PAREADA_C4, AZAR_C4, "C4")
if INCLUIR_C5:
    CONCLUSIONES += [f"[C5] {t}" for t in rv.conclusiones(ORDEN.iloc[0:0], RESUMEN_C5, PAREADA_C5,
                                                         AZAR_C4, "C5")[:2]]
for frase in CONCLUSIONES:
    print("•", frase)

# Por cohorte solo se publica el riesgo relativo: los conteos permitirían deducir celdas suprimidas
rv.para_publicar(SENALES, ["C1–C3", "C2–C3 (entrenamiento)"]).to_csv(
    MET / f"revision_senales_{FUENTE}.csv", index=False)
ORDEN.to_csv(MET / f"revision_orden_senales_{FUENTE}.csv", index=False)
SORTEOS_C4.describe().round(4).to_csv(MET / f"revision_sorteos_C4_{FUENTE}.csv")
DESCRIPTIVOS.to_csv(MET / f"descriptivos_{FUENTE}.csv", index=False)
resumen = {"fuente": FUENTE, "k_por_sede": K, "sorteos": SORTEOS, "repeticiones": REPETICIONES,
           "incluye_C5": INCLUIR_C5, "orden_senales": ORDEN.to_dict(orient="records"),
           "sorteos_C4": RESUMEN_C4, "pareada_D2_menos_D_C4": PAREADA_C4, "azar_C4": AZAR_C4,
           "sorteos_C5": RESUMEN_C5, "pareada_D2_menos_D_C5": PAREADA_C5,
           "conclusiones": CONCLUSIONES}
(MET / f"revision_{FUENTE}.json").write_text(
    json.dumps(resumen, indent=2, ensure_ascii=False, default=float), encoding="utf-8")
print("\nGuardado: revision_senales, revision_orden_senales, revision_sorteos_C4, descriptivos y revision_" + FUENTE + ".json")

In [ ]:
# @title Descargar agregados para compartir (Colab)
# Solo métricas agregadas: no contiene datos individuales.
import shutil
paquete = RAIZ / f"revision_{FUENTE}"
shutil.rmtree(paquete, ignore_errors=True); paquete.mkdir()
for nombre in [f"revision_senales_{FUENTE}.csv", f"revision_orden_senales_{FUENTE}.csv",
               f"revision_sorteos_C4_{FUENTE}.csv", f"descriptivos_{FUENTE}.csv",
               f"revision_{FUENTE}.json"]:
    shutil.copy(MET / nombre, paquete)
zip_final = shutil.make_archive(str(paquete), "zip", paquete)
shutil.rmtree(paquete)
print("Contenido:", sorted(zipfile.ZipFile(zip_final).namelist()))
if EN_COLAB:
    from google.colab import files
    files.download(zip_final)